In [ ]:
!pip install comet_ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.6 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.4.1
    Uninstalling python-box-7.4.1:
      Successfully uninstalled python-box-7.4.1


In [ ]:
!git clone -b lensless-base https://github.com/DommeUse/Lensless-Computational-Imaging.git
%cd Lensless-Computational-Imaging
!python -m pip install virtualenv
!python -m virtualenv /content/lensless_env
!/content/lensless_env/bin/pip install -r requirements.txt

Cloning into 'Lensless-Computational-Imaging'...
remote: Enumerating objects: 665, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 665 (delta 130), reused 124 (delta 63), pack-reused 420 (from 1)
Receiving objects: 100% (665/665), 336.98 KiB | 4.26 MiB/s, done.
Resolving deltas: 100% (304/304), done.
/content/Lensless-Computational-Imaging
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 43.0 MB/s eta 0:00:00
created virtual environment CPython3.12.13.final.0-64-x86_64 in 321ms
  creator CPython3Posix(dest=/content/lensless_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.1.2
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator


In [ ]:
import os
from google.colab import userdata

os.environ["COMET_API_KEY"] = userdata.get("COMET_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["MPLBACKEND"] = "Agg"

# Unrolled ADMM-20

In [ ]:
!/content/lensless_env/bin/python -u train.py \
  --config-name=admm_unrolled \
  trainer.save_dir="/content/saved" \
  writer.log_checkpoints=True \
  writer.run_name="Unrolled ADMM train"

Logging git commit and patch...
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: Experiment is live on comet.com https://www.comet.com/german-zverev/lensless-computational-imaging/iirofhqpyu4mfudzv34obsgvvm1uheq9















ADMM()
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth























Train Epoch: 1 [0/2125 (0%)] Loss: 0.931599






















































Train Epoch: 1 [50/2125 (2%)] Loss: 0.937558









Train Epoch: 1 [100/2125 (5%)] Loss: 0.896639
Train Epoch: 1 [150/2125 (7%)] Loss: 0.960394
Train Epoch: 1 [200/2125 (9%)] Loss: 1.117046
Train Epoch: 1 [250/2125 (12%)] Loss: 0.856178
Train Epoch: 1 [300/2125 (14%)] Loss: 0.909557
Train Epoch: 1 [350/2125 (16%)] Loss: 0.910421
Train Epoch: 1 [400/2125 (19%)] Loss: 1.056449
Train Epoch: 1 [450/2125 (21%)] Loss: 0.963449
Train Epoch: 1 [500/2

# I. LeADMM-5. Pre- and Post-processors

In [ ]:
import comet_ml

api = comet_ml.API(api_key=userdata.get("COMET_API_KEY"))
exp = api.get_experiment("german-zverev", "lensless-computational-imaging", "sobsv72s4yhtocgfiagbym6u40z6qfef")

assets = exp.get_asset_list()
model_asset = [a for a in assets if "checkpoint-epoch10" in a['fileName']]

asset_id = model_asset[0]['assetId']
asset_binary = exp.get_asset(asset_id)

with open("./model_best.pth", "wb") as f:
    f.write(asset_binary)

import torch

checkpoint = torch.load("./model_best.pth", "cuda", weights_only = False)
checkpoint['config']['writer']['run_id'] = 'sobsv72s4yhtocgfiagbym6u40z6qfef'
torch.save(checkpoint, './model_best.pth')

In [ ]:
checkpoint = torch.load("./model_best.pth", "cuda", weights_only = False)

In [ ]:
checkpoint['config']['writer']['run_id']

'sobsv72s4yhtocgfiagbym6u40z6qfef'

In [ ]:
!/content/lensless_env/bin/python -u train.py \
  --config-name=modular \
  model=modular_pre_post \
  trainer.save_dir="/content/saved" \
  trainer.max_grad_norm=1.0 \
  writer.log_checkpoints=True \
  writer.run_name="Modular-Pre-Post train" \
  +writer.run_id="sobsv72s4yhtocgfiagbym6u40z6qfef" \
  trainer.resume_from="./model_best.pth"

Logging git commit and patch...
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: Experiment is live on comet.com https://www.comet.com/german-zverev/lensless-computational-imaging/sobsv72s4yhtocgfiagbym6u40z6qfef















ModularReconstruction(
  (pre): DRUNet(
    (input_conv): Conv2d(3, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_block1): DownBlock(
      (res_blocks): Sequential(
        (0): ResBlock(
          (conv1): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (act): ReLU()
          (conv2): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (1): ResBlock(
          (conv1): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (act): ReLU()
          (conv2): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (2): ResBlock(
          (conv1): Conv2d(24, 24, kernel_

# II. LeADMM-5. Only Pre-processor

In [ ]:
!/content/lensless_env/bin/python -u train.py \
  --config-name=modular \
  model=modular_pre \
  trainer.save_dir="/content/saved" \
  trainer.max_grad_norm=1.0 \
  writer.log_checkpoints=True \
  writer.run_name="Modular-Pre train"

Logging git commit and patch...
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: Experiment is live on comet.com https://www.comet.com/german-zverev/lensless-computational-imaging/nzgn47ok8tqnwlz7lme6d6nasltaitwz















ModularReconstruction(
  (pre): DRUNet(
    (input_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_block1): DownBlock(
      (res_blocks): Sequential(
        (0): ResBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (act): ReLU()
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (1): ResBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (act): ReLU()
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (2): ResBlock(
          (conv1): Conv2d(32, 32, kernel_

# III. LeADMM-5. Only Post-processor

In [ ]:
!/content/lensless_env/bin/python -u train.py \
  --config-name=modular \
  model=modular_post \
  trainer.save_dir="/content/saved" \
  trainer.max_grad_norm=1.0 \
  writer.log_checkpoints=True \
  writer.run_name="Modular-Post train"

Logging git commit and patch...
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: Experiment is live on comet.com https://www.comet.com/german-zverev/lensless-computational-imaging/a5z5pkiyealbs5c58giiw2slih9v4vho















ModularReconstruction(
  (admm): ADMM()
  (post): DRUNet(
    (input_conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_block1): DownBlock(
      (res_blocks): Sequential(
        (0): ResBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (act): ReLU()
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (1): ResBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (act): ReLU()
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (2): ResBlock(
          (conv1): Conv